In [2]:
!apt-get install tesseract-ocr -y
!pip install pytesseract pdf2image pillow pandas google-generativeai

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [3]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from google.colab import files
import pandas as pd
import json
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
genai.configure(api_key="")

model = genai.GenerativeModel("gemini-2.5-flash")

In [28]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving HCFA1500.jpg to HCFA1500 (1).jpg


In [29]:
def load_images(path):

    if path.lower().endswith(".pdf"):
        return convert_from_path(path)
    else:
        return [Image.open(path)]

images = load_images(file_path)

In [30]:
full_text = ""

for img in images:
    text = pytesseract.image_to_string(img)
    full_text += text + "\n"

print(full_text[:2000])   # preview

1

2

5

6

workplacehcfa1 500

visit: www.paris-software.com/samples/workplacehcfa1 500

 

PLEASE eee MEDICARE 0555 Pea
a appr ba
SPE 1212 HOSPITAL WAY pete
hy Fe ey WING C 92 ote atta
AREA Das LOS ANGELES CA 90000 aes
HEALTH INSURANCE CLAIM FORM
Is MEDICARE MEDICAID CHAMPUS CHAMPVA GROUP FEC, OTHER] 1A. INSURED’S |.D. NUMBER (FOR PROGRAM IN ITEM 1)

A
EALTH PLAN BLK LUNG

H
(Medicare #) a (Medicaid #) a (Sponsor's SSN) | Dn Alen a ieee U2) a veel | | (1D) 3 3 3 3 3 3 3 3
SEX

2. PATIENT S NAME (Last Name, First Name, Middle Initial) ore BIRTH DATE 4. INSURED'S NAME (Last Name, First Name, Middle Initial)
DD ¥Y
SMITH JOHN C\02 14 00 | | =| X}| SMITH JOHN
5. PATIENT S ADDRESS (No Street) 6. PATIENT S RELATIONSHIP TO INSURED 7. INSURED'S ADDRESS (No Street)
ooo RIDGE RD sett} XJ Spouse] fori} | her! =| | 555 RIDGE ST

             
   

CITY STATE | 8. PATIENT STATUS CITY STATE
SAN DIEGO CA | sive[ | wewieaf] one []] SAN DIEGO CA
ZIP CODE TELEPHONE (Include Area Code) TELEPHONE (INCLUD

In [31]:
prompt = f"""
You are an expert healthcare claim data extraction system.

Your task is to extract structured information from OCR text of a US healthcare claim form.

IMPORTANT RULES:
1. Extract ONLY the requested fields.
2. DO NOT guess or invent values.
3. If a field is missing, return null.
4. Output MUST be valid JSON.
5. Do NOT include explanations, comments, or markdown.
6. Normalize values when possible:
   - Dates → YYYY-MM-DD
   - Amounts → numbers only (no $ or commas)
7. If multiple values exist, choose the most relevant claim-level value.
8. In diagnosis code , if it not an alphanumeric number then remove the decimal

FIELDS TO EXTRACT:
- patient_name
- member_id
- provider_name
- date_of_service
- diagnosis_code
- procedure_code
- charge_amount

Return EXACTLY this JSON structure:

{{
  "patient_name": null,
  "member_id": null,
  "provider_name": null,
  "date_of_service": null,
  "diagnosis_code": null,
  "procedure_code": null,
  "charge_amount": null
}}

Claim Text:
--------------------
{full_text}
--------------------
"""

In [36]:
response = model.generate_content(prompt)

structured_output = response.text

print(structured_output)

```json
{
  "patient_name": "JOHN C SMITH",
  "member_id": "3333333333",
  "provider_name": "DLLISON PAYLOR PLAY COMPANY",
  "date_of_service": "2006-01-07",
  "diagnosis_code": "4019",
  "procedure_code": "00180",
  "charge_amount": 1500.00
}
```


In [38]:
clean = structured_output.replace("```json","").replace("```","")

data = json.loads(clean)

print(data)

{'patient_name': 'JOHN C SMITH', 'member_id': '3333333333', 'provider_name': 'DLLISON PAYLOR PLAY COMPANY', 'date_of_service': '2006-01-07', 'diagnosis_code': '4019', 'procedure_code': '00180', 'charge_amount': 1500.0}


In [39]:
print("Raw CPT from LLM:", data.get("procedure_code"))

Raw CPT from LLM: 00180


In [40]:
def normalize_cpt(code):

    if code is None:
        return None

    code = str(code).lower().strip()

    replacements = {
        "o": "0",
        "g": "9",
        "t": "1",
        "l": "1",
        "i": "1",
        "s": "5",
        "b": "8"
    }

    corrected = ""

    for ch in code:
        if ch.isdigit():
            corrected += ch
        elif ch in replacements:
            corrected += replacements[ch]

    corrected = "".join(c for c in corrected if c.isdigit())

    if len(corrected) == 5:
        return corrected

    return None

In [41]:
data["procedure_code"] = normalize_cpt(
    data.get("procedure_code")
)

print("Corrected CPT:", data["procedure_code"])

Corrected CPT: 00180


In [13]:
def excel_safe(value):
    if pd.isna(value):
        return value
    return f'="{value}"'

In [14]:
df = pd.DataFrame([data])

protect_cols = [
    "procedure_code",
    "diagnosis_code",
    "member_id"
]

for col in protect_cols:
    if col in df.columns:
        df[col] = df[col].apply(excel_safe)

df.to_csv("claim_output.csv", index=False)

In [16]:
df=pd.read_csv("claim_output.csv")
df

,patient_name,member_id,provider_name,date_of_service,diagnosis_code,procedure_code,charge_amount
0,SMITH JOHN C,"=""33333333""",OFFICE OF DLLISON PAYLOR PLAY COMPANY,2006-11-01,"=""401.9""","=""00100""",1500


In [18]:
files.download("claim_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>